# 6. Deployment: Servis Inferensi FastAPI & Kontainerisasi Docker

Tahap **Deployment** menyediakan microservice REST API produksi berbasis **FastAPI + Uvicorn** pada **Port `8016`** untuk memprediksi status kerapian citra sandal & sepatu secara waktu-nyata:

1. **Microservice REST API (`app.py`):**
   * `POST /api/v1/predict`: Menerima unggahan foto (JPEG/PNG), menjaga orientasi EXIF asli, mengekstrak **65 fitur visual tahan lantai**, menerapkan `StandardScaler`, dan mengembalikan kelas prediksi beserta tingkat keyakinan (*confidence score*) dan latensi inferensi.
   * `GET /health`: Endpoint pemantauan status server dan kesiapan bobot model.
   * `GET /`: Antarmuka Web Interaktif (*Dark Theme*) untuk pengujian langsung melalui browser.
2. **Kontainerisasi Docker:** Dilengkapi `Dockerfile` dan `docker-compose.yml` untuk portabilitas deployment.
3. **Verifikasi Inferensi End-to-End:** Pengujian langsung terhadap sampel citra dari **`2-data_understanding/data/`** menggunakan pipeline deployment.


In [1]:
import sys
from pathlib import Path
from PIL import Image


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "src" / "10-project_citra"):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR / "2-data_understanding" / "data"
DEPLOYMENT_DIR = PROJECT_DIR / "6-deployment"
sys.path.insert(0, str(DEPLOYMENT_DIR))

from app import extract_features_from_image, model, scaler

print("Model Loaded  :", model is not None, f"({type(model).__name__})")
print("Scaler Loaded :", scaler is not None)
if model is None or scaler is None:
    raise RuntimeError("Model/scaler deployment belum tersedia. Jalankan run_pipeline.py terlebih dahulu.")

print("\n=== Uji Inferensi Sampel Aktual ===")
for folder_name, label in (("RAPI", "Rapi"), ("BERANTAKAN", "Berantakan")):
    candidates = sorted(
        path for path in (DATASET_DIR / folder_name).rglob("*")
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not candidates:
        raise FileNotFoundError(f"Tidak ada citra di folder {folder_name}.")
    k2_candidates = [p for p in candidates if "kelompok 2" in str(p.parent).lower()]
    sample_path = k2_candidates[0] if k2_candidates else candidates[0]
    with Image.open(sample_path) as image:
        features = extract_features_from_image(image)
    scaled_features = scaler.transform([features])
    prediction = int(model.predict(scaled_features)[0])
    class_probabilities = {
        int(class_label): float(probability)
        for class_label, probability in zip(model.classes_, model.predict_proba(scaled_features)[0])
    }
    prediction_text = "Rapi" if prediction == 1 else "Berantakan"
    print(
        f"{sample_path.parent.name}/{sample_path.name} -> "
        f"Aktual={label}, Prediksi={prediction_text}, Prob_Rapi={class_probabilities[1]:.3f}"
    )


Model, scaler, and feature definitions loaded successfully.
Model Loaded  : True (LGBMClassifier)
Scaler Loaded : True

=== Uji Inferensi Sampel Aktual ===


Kelompok 2/IMG_20261104_122311.jpg -> Aktual=Rapi, Prediksi=Rapi, Prob_Rapi=0.968
Kelompok 2/IMG_20261104_122321.jpg -> Aktual=Berantakan, Prediksi=Rapi, Prob_Rapi=0.520


---
## 6.1 Panduan Menjalankan Layanan Deployment

Jalankan server API FastAPI secara lokal melalui terminal:
```bash
cd "src/10-project_citra/6-deployment"
python -m uvicorn app:app --host 0.0.0.0 --port 8016 --reload
```
Akses dashboard web interaktif di `http://localhost:8016/` atau dokumentasi Swagger API di `http://localhost:8016/docs`.
